# 原油SC + PTA投资分析报告

This notebook provides a complete analysis of the energy chemical theme: crude oil (SC) and PTA price relationship, volatility, and risk characteristics.

Sections:
1. Load data
2. Data cleaning and preprocessing
3. Data analysis and visualization
4. Report summary and export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = Path('data_clean/final.csv')

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

df.head()

In [ ]:
# 数据基本信息
print('样本数量：', len(df))
print('时间区间：', df['date'].min().strftime('%Y-%m-%d'), '到', df['date'].max().strftime('%Y-%m-%d'))
print('字段：', list(df.columns))

# 样本描述统计
summary = df[['SC_close','PTA_close','SC_ret','PTA_ret','SC_vol20','PTA_vol20']].describe().round(4)
summary

## 数据清洗与预处理

本节检查数据完整性，确认缺失值、重复值和数值范围是否符合分析要求。

In [ ]:
# 检查缺失值与重复值
missing = df.isnull().sum()
duplicates = df.duplicated(subset=['date']).sum()
print('缺失值统计：\n', missing)
print('重复日期数量：', duplicates)

# 检查价格范围
price_ranges = df[['SC_close','PTA_close']].agg(['min','max']).round(4)
price_ranges

## 数据分析与可视化

本节展示价格趋势、收益率分布、波动率趋势和相关性分析。

In [ ]:
# 价格趋势比较
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['SC_close'], label='SC Close', color='#1f77b4', linewidth=1.2)
plt.plot(df['date'], df['PTA_close'], label='PTA Close', color='#ff7f0e', linewidth=1.2)
plt.title('原油 SC 与 PTA 价格趋势')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 收益率分布与核密度
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['SC_ret'], bins=80, color='#1f77b4', alpha=0.7, density=True)
axes[0].set_title('SC 收益率分布')
axes[0].grid(alpha=0.3)

axes[1].hist(df['PTA_ret'], bins=80, color='#ff7f0e', alpha=0.7, density=True)
axes[1].set_title('PTA 收益率分布')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(10,5))
sns.kdeplot(df['SC_ret'], fill=True, label='SC', color='#1f77b4')
sns.kdeplot(df['PTA_ret'], fill=True, label='PTA', color='#ff7f0e')
plt.title('SC 与 PTA 收益率核密度')
plt.xlabel('Log Return')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 波动率趋势和相关性分析
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['SC_vol20'], label='SC 20 日波动率', color='#1f77b4')
plt.plot(df['date'], df['PTA_vol20'], label='PTA 20 日波动率', color='#ff7f0e')
plt.title('SC 与 PTA 20 日滚动波动率')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr_price = df['SC_close'].corr(df['PTA_close'])
corr_ret = df['SC_ret'].corr(df['PTA_ret'])
corr_vol = df['SC_vol20'].corr(df['PTA_vol20'])

print(f'价格相关系数 (SC vs PTA): {corr_price:.4f}')
print(f'收益率相关系数 (SC vs PTA): {corr_ret:.4f}')
print(f'波动率相关系数 (SC vs PTA): {corr_vol:.4f}')

plt.figure(figsize=(6,6))
sns.scatterplot(x=df['SC_ret'], y=df['PTA_ret'], alpha=0.4)
plt.title('SC 与 PTA 日收益率散点图')
plt.xlabel('SC 收益率')
plt.ylabel('PTA 收益率')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 极端波动统计
sc_extreme = (df['SC_ret'].abs() > 0.03).mean() * 100
pta_extreme = (df['PTA_ret'].abs() > 0.03).mean() * 100
print(f'SC 极端波动比例 (>±3%): {sc_extreme:.2f}%')
print(f'PTA 极端波动比例 (>±3%): {pta_extreme:.2f}%')


## 生成报告输出

本节将分析结论导出为文本报告，并总结关键结论与投资建议。

In [ ]:
summary_text = f'''
数据分析结论
===============

1. 价格相关性：SC 与 PTA 收盘价相关系数为 {corr_price:.4f}，表明长期价格趋势高度一致。
2. 收益率相关性：日收益率相关系数为 {corr_ret:.4f}，显示短期传导中等但仍具有同步性。
3. 波动率相关性：20 日波动率相关系数为 {corr_vol:.4f}，说明两者风险水平同步。
4. 极端风险：SC 发生极端波动的频率为 {sc_extreme:.2f}%，PTA 为 {pta_extreme:.2f}%。

建议：
- 当 SC 价格趋势明确时，可参考其信号选择 PTA 趋势仓位。
- 在原油高波动阶段，建议对下游 PTA 成本进行套期保值。
- 通过 SC/PTA 相关性设计跨品种套利策略具有可行性。
'''

with open('report_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print('报告已导出到 report_summary.txt')
print(summary_text)